# Constraints, Primary Keys & Identity in SQL Server

In this section we will understand **why constraints are essential**, how they protect data integrity, and why SQL Server uses the concept of **Identity (Surrogate Key)**.

We will also solve a real problem you just faced:  
> “I inserted two identical rows and now I cannot delete only one of them.”

## 1. The Real Problem: Duplicate Rows

When you insert two completely identical rows into a table that has **no Primary Key** or **Unique constraint**, SQL Server cannot distinguish between them.

As a result:
- You cannot delete only one of them using the visual editor.
- A simple `DELETE` without a precise `WHERE` clause will delete **both** rows.

This happens because SQL Server has no unique identifier to target a specific row.

## 2. What Are Constraints?

**Constraints** are rules that SQL Server enforces on columns or tables to protect data quality and integrity.

They prevent invalid, incomplete, or inconsistent data from being stored.

## 3. Main Types of Constraints in SQL Server

| Constraint Type       | Purpose                                      | Example |
|-----------------------|----------------------------------------------|---------|
| **Data Type**         | Restricts the kind of data a column can store | `INT`, `CHAR(10)`, `DATE` |
| **NOT NULL**          | Column cannot be empty                       | `FirstName NVARCHAR(50) NOT NULL` |
| **UNIQUE**            | Values must be unique (NULL is allowed)      | `StudentNumber VARCHAR(15) UNIQUE` |
| **PRIMARY KEY**       | Unique + NOT NULL + main identifier of the table | `StudentID INT PRIMARY KEY` |
| **CHECK**             | Enforces a logical condition                 | `CHECK (Gender IN ('M','F'))` |
| **FOREIGN KEY**       | Creates relationship with another table      | `FOREIGN KEY (CourseID) REFERENCES Course(CourseID)` |
| **DEFAULT**           | Provides a default value                     | `DEFAULT GETDATE()` |

## 4. Characteristics of a Good Primary Key

A good Primary Key should have these properties:

1. **NOT NULL** – It can never be empty
2. **Unique** – No two rows can have the same value
3. **Stable** – It should not change frequently (avoid updating it)
4. **Narrow** – Prefer small data types (better performance for indexes and joins)
5. Helps maintain **Data Consistency** and **Entity Integrity**
6. Should be chosen from the available **Candidate Keys**

## 5. Candidate Keys

A **Candidate Key** is any column (or set of columns) that can uniquely identify a row.

In our `Student` table we have two natural candidate keys:

- `StudentNumber`
- `NationalCode`

Both are unique and not null.  
We can choose one of them as the Primary Key, or we can create a new artificial key (Surrogate Key).

## 6. Why Didn’t We Make Mobile Number Unique?

At this stage of the design we intentionally did **not** put a UNIQUE constraint on the phone number.

**Reason:**
- Sometimes parents register their children.
- The same mobile number may be used for multiple students.
- Therefore, mobile number is **not** a reliable unique identifier in this business scenario.

## 7. Complete Student Table Example

In [ ]:
USE SematecLMS;
GO

CREATE TABLE Student
(
    StudentID      INT IDENTITY(1,1) PRIMARY KEY,          -- Surrogate Key
    StudentNumber  VARCHAR(15)  NOT NULL UNIQUE,           -- Natural Candidate Key
    NationalCode   CHAR(10)     NOT NULL UNIQUE            -- Natural Candidate Key
                   CHECK (LEN(NationalCode) = 10),
    FirstNameEn    VARCHAR(50)  NOT NULL,
    LastNameEn     VARCHAR(50)  NOT NULL,
    FirstNameFa    NVARCHAR(50) COLLATE Persian_100_CI_AS NOT NULL,
    LastNameFa     NVARCHAR(50) COLLATE Persian_100_CI_AS NOT NULL,
    Gender         CHAR(1)      NOT NULL
                   CHECK (Gender IN ('M','F')),
    BirthDate      DATE         NOT NULL
                   CHECK (BirthDate <= GETDATE()),
    Education      NVARCHAR(100) NULL,
    Job            NVARCHAR(100) NULL,
    FieldOfStudy   NVARCHAR(100) NOT NULL,
    Address        NVARCHAR(500) NULL,
    PhoneNumber    VARCHAR(15)   NULL,
    Picture        VARBINARY(MAX) NULL
);

## 8. Why Do We Use a Surrogate Key (Identity)?

Natural keys (like National Code or Student Number) are good, but they have some disadvantages:

- They might change in the future
- They are sometimes wide (CHAR(10), VARCHAR(15))
- Combining multiple columns as a key makes joins slower

Microsoft (and most database systems) recommend using a **Surrogate Key**:

- An artificial column (usually named `ID` or `StudentID`)
- Automatically generated by SQL Server
- Narrow data type (`INT` or `BIGINT`)
- Never changes

## 9. Understanding IDENTITY

```sql
StudentID INT IDENTITY(1,1) PRIMARY KEY

Seed = Starting value (default is 1)
Increment = How much to increase each time (default is 1)

So the first row gets 1, the second gets 2, the third gets 3, and so on.
You almost never need to change the default values (1,1).



## 10. Can We Always Add an ID Column?

**Yes.**

Even if a table already has 1,000 rows, you can still add an `IDENTITY` column.  
SQL Server will automatically generate values for the existing rows.

However, it is much cleaner to design the table with the Identity column from the beginning.

## 11. Can a Table Exist Without an ID?

**Yes**, there are valid exceptions.

The most common example is a **Junction (Bridge) Table**:

```sql
CREATE TABLE StudentCourse
(
    StudentID INT NOT NULL,
    CourseID  INT NOT NULL,
    EnrollmentDate DATE NOT NULL DEFAULT GETDATE(),

    CONSTRAINT PK_StudentCourse PRIMARY KEY (StudentID, CourseID)
);

In this case, the combination of StudentID + CourseID is the Primary Key.

A separate surrogate key is usually unnecessary.
text


---
## 12. Why Do We Prefer INT for Identity Columns?

| Data Type   | Approximate Range     | Recommended Use              |
|-------------|-----------------------|------------------------------|
| `SMALLINT`  | ± 32,000        | Only very small tables       |
| `INT`       | Over 2 billion        | **Standard choice**          |
| `BIGINT`    | Extremely large       | Very high-volume systems     |

`INT` is the industry standard because the chance of running out of values is extremely low for most applications.

## 13. How to Set Identity in SSMS (Table Designer)

1. Right-click the table → **Design**
2. Select the `StudentID` column
3. In **Column Properties** (bottom panel):
   - Expand **Identity Specification**
   - Set `(Is Identity)` = `Yes`
   - `Identity Seed` = `1`
   - `Identity Increment` = `1`
4. Save the table

Alternative way:
- Right-click table → **Properties** → **Identity Column**

## 14. Important Design Rule

The **Surrogate Key** (`StudentID`) is an **internal** technical identifier.

- End users should **never** see or enter this value.
- Users work with business keys such as `StudentNumber` or `NationalCode`.
- The Identity value is only used by the database and by developers for joins and relationships.

## 15. Common Problem: Identity Starts from 2

This is one of the most frequent questions from beginners.

### Why does it happen?

- You tried to insert a row and it **failed** (error occurred)
- SQL Server still consumed the Identity value
- You deleted a row
- The table was dropped and recreated without resetting the Identity

### How to Fix It

```sql
-- Check current Identity value
DBCC CHECKIDENT ('Student');

-- Reset so the next value will be 1
DBCC CHECKIDENT ('Student', RESEED, 0);

DBCC CHECKIDENT ('Student', RESEED, 99);


## 16. Summary

| Concept                  | Key Point                                      |
|--------------------------|------------------------------------------------|
| Duplicate rows problem   | Happens when there is no Primary Key / Unique  |
| Candidate Keys           | `StudentNumber` and `NationalCode`             |
| Surrogate Key            | `StudentID INT IDENTITY(1,1)`                  |
| Why Identity?            | Stable, narrow, automatic, simple              |
| Seed & Increment         | Default is `(1,1)`                             |
| Tables without ID        | Possible (especially Junction tables)          |
| Identity starts from 2   | Usually caused by a failed INSERT → use RESEED |
| Best Practice            | Always design tables with a proper Primary Key |

## Practice

1. Create the `Student` table using the script above.
2. Insert 3 rows (let Identity generate the IDs).
3. Try to insert a duplicate `NationalCode` and observe the error.
4. Delete one row and check the next Identity value.
5. Use `DBCC CHECKIDENT` to reset the Identity.